# Herança, exceções e projeto orientado a objetos

Este notebook junta três assuntos que aparecem bastante quando a gente começa a sair dos exemplos muito pequenos de POO:

- herança e tratamento de exceções;
- análise orientada a objetos;
- projeto orientado a objetos.

O foco maior fica no primeiro tema, porque herança e exceções costumam ser onde o código começa a ficar mais interessante. Para deixar menos seco, o exemplo vai usar uma quitanda meio bagunçada, com personagens como Zé da Manga, Zé da Couve e Dona Jurema do Abacaxi.

## 1. Herança e tratamento de exceções

Herança serve para reaproveitar uma ideia geral e criar versões mais específicas dela. No nosso caso, todo produto tem nome, preço e estoque. Mas uma manga vendida na quitanda pode ter nível de maturação, enquanto uma couve pode ter quantidade de maços.

Tratamento de exceções entra quando alguma regra do sistema é quebrada. Em vez de deixar o programa explodir com um erro genérico, criamos mensagens mais claras e controlamos melhor o problema.

In [ ]:
class EstoqueInsuficienteError(Exception):
    """Erro usado quando alguém tenta vender mais do que existe no estoque."""
    pass


class PrecoInvalidoError(Exception):
    """Erro usado quando o preço não faz sentido para o produto."""
    pass


class Produto:
    def __init__(self, nome, preco, estoque):
        if preco <= 0:
            raise PrecoInvalidoError(f"Preço inválido para {nome}: {preco}")
        if estoque < 0:
            raise ValueError("Estoque não pode ser negativo.")

        self.nome = nome
        self.preco = preco
        self.estoque = estoque

    def vender(self, quantidade):
        if quantidade <= 0:
            raise ValueError("Quantidade vendida precisa ser maior que zero.")
        if quantidade > self.estoque:
            raise EstoqueInsuficienteError(
                f"Só tem {self.estoque} unidade(s) de {self.nome}. Zé da Manga tentou forçar a amizade."
            )

        self.estoque -= quantidade
        return quantidade * self.preco

    def repor(self, quantidade):
        if quantidade <= 0:
            raise ValueError("Reposição precisa ser maior que zero.")
        self.estoque += quantidade

    def descricao(self):
        return f"{self.nome} | R$ {self.preco:.2f} | estoque: {self.estoque}"

    def __str__(self):
        return self.descricao()

A classe `Produto` é a classe base. Ela guarda o que todo produto precisa ter. Agora entram as subclasses, que herdam o comportamento geral e adicionam detalhes próprios.

In [ ]:
class Fruta(Produto):
    def __init__(self, nome, preco, estoque, maturacao):
        super().__init__(nome, preco, estoque)
        self.maturacao = maturacao

    def descricao(self):
        texto_base = super().descricao()
        return f"{texto_base} | maturação: {self.maturacao}"


class Verdura(Produto):
    def __init__(self, nome, preco, estoque, unidade):
        super().__init__(nome, preco, estoque)
        self.unidade = unidade

    def descricao(self):
        texto_base = super().descricao()
        return f"{texto_base} | vendida por: {self.unidade}"


class ProdutoPromocional(Produto):
    def __init__(self, nome, preco, estoque, desconto):
        super().__init__(nome, preco, estoque)
        if not 0 < desconto < 1:
            raise ValueError("Desconto precisa ficar entre 0 e 1.")
        self.desconto = desconto

    def vender(self, quantidade):
        total_normal = super().vender(quantidade)
        return total_normal * (1 - self.desconto)

    def descricao(self):
        texto_base = super().descricao()
        return f"{texto_base} | desconto: {self.desconto:.0%}"

Aqui aparecem três pontos importantes:

- `Fruta`, `Verdura` e `ProdutoPromocional` são tipos específicos de `Produto`;
- `super()` chama o comportamento da classe mãe;
- `ProdutoPromocional` sobrescreve o método `vender`, porque a regra de venda dela é diferente.

In [ ]:
manga = Fruta("Manga do Zé da Manga", 4.50, 8, "quase no ponto")
couve = Verdura("Couve do Zé da Couve", 3.00, 5, "maço")
abacaxi = ProdutoPromocional("Abacaxi da Dona Jurema", 7.00, 3, 0.15)

produtos = [manga, couve, abacaxi]

for produto in produtos:
    print(produto.descricao())

O loop acima também mostra polimorfismo: todos os objetos respondem a `descricao()`, mas cada classe monta a descrição do seu jeito.

In [ ]:
try:
    total = manga.vender(2)
    print(f"Venda feita. Total: R$ {total:.2f}")
    print(manga)
except EstoqueInsuficienteError as erro:
    print("Problema de estoque:", erro)
except ValueError as erro:
    print("Valor estranho informado:", erro)
finally:
    print("Conferência da venda finalizada.")

In [ ]:
try:
    total = couve.vender(20)
    print(f"Venda feita. Total: R$ {total:.2f}")
except EstoqueInsuficienteError as erro:
    print("Deu ruim no estoque:", erro)
except ValueError as erro:
    print("Entrada inválida:", erro)

Neste ponto dá para perceber uma diferença importante: erro não é sempre sinal de código mal feito. Às vezes o erro representa uma situação prevista do negócio. O estoque acabar é normal. O que não pode é o sistema responder de qualquer jeito.

In [ ]:
try:
    produto_esquisito = Produto("Banana invisível do Seu Ninguém", -2.00, 10)
except PrecoInvalidoError as erro:
    print("Cadastro recusado:", erro)

## 2. Análise orientada a objetos

Antes de sair criando classe, vale pensar no problema. Uma análise simples para a quitanda poderia ser assim:

- Quem são os objetos principais? Produto, Cliente, Pedido e ItemPedido.
- O que cada objeto sabe? Produto sabe preço e estoque. Cliente sabe nome. Pedido sabe quem comprou e quais itens entraram.
- O que cada objeto faz? Produto vende/repoe. Pedido calcula total. ItemPedido calcula subtotal.

Essa parte é menos sobre código e mais sobre enxergar responsabilidades. Se uma classe sabe coisa demais, normalmente ela começa a virar bagunça.

In [ ]:
class Cliente:
    def __init__(self, nome, apelido):
        self.nome = nome
        self.apelido = apelido

    def __str__(self):
        return f"{self.nome} ({self.apelido})"


class ItemPedido:
    def __init__(self, produto, quantidade):
        self.produto = produto
        self.quantidade = quantidade

    def subtotal(self):
        return self.produto.preco * self.quantidade


class Pedido:
    def __init__(self, cliente):
        self.cliente = cliente
        self.itens = []

    def adicionar_item(self, produto, quantidade):
        self.itens.append(ItemPedido(produto, quantidade))

    def total_previsto(self):
        return sum(item.subtotal() for item in self.itens)

    def fechar(self):
        total = 0
        for item in self.itens:
            total += item.produto.vender(item.quantidade)
        return total

In [ ]:
cliente = Cliente("José Marinaldo", "Zé da Manga")
pedido = Pedido(cliente)

pedido.adicionar_item(manga, 2)
pedido.adicionar_item(abacaxi, 1)

print("Cliente:", pedido.cliente)
print(f"Total previsto sem regra promocional: R$ {pedido.total_previsto():.2f}")

try:
    total_final = pedido.fechar()
    print(f"Total final considerando as regras das classes: R$ {total_final:.2f}")
except EstoqueInsuficienteError as erro:
    print("Pedido não fechado:", erro)

## 3. Projeto orientado a objetos

Na parte de projeto, a ideia é organizar as classes para facilitar manutenção. Algumas decisões usadas aqui:

- Produto fica como classe geral, porque fruta, verdura e promoção compartilham regras.
- Exceções próprias deixam claro o tipo de problema.
- Pedido não altera estoque diretamente. Ele pede para o produto vender.
- ItemPedido existe para não colocar produto e quantidade soltos dentro de uma lista.

Isso ajuda porque cada classe fica com uma função mais clara. Quando aparecer uma regra nova, como produto congelado ou desconto por cliente antigo, existe um lugar mais natural para mexer.

## A mesma ideia em JavaScript

JavaScript também tem classes, herança com `extends`, chamada da classe mãe com `super()` e tratamento de exceções com `try`, `catch` e `finally`. A sintaxe muda, mas a lógica principal é bem parecida.

```javascript
class EstoqueInsuficienteError extends Error {
  constructor(message) {
    super(message);
    this.name = "EstoqueInsuficienteError";
  }
}

class PrecoInvalidoError extends Error {
  constructor(message) {
    super(message);
    this.name = "PrecoInvalidoError";
  }
}

class Produto {
  constructor(nome, preco, estoque) {
    if (preco <= 0) {
      throw new PrecoInvalidoError(`Preço inválido para ${nome}: ${preco}`);
    }
    if (estoque < 0) {
      throw new Error("Estoque não pode ser negativo.");
    }

    this.nome = nome;
    this.preco = preco;
    this.estoque = estoque;
  }

  vender(quantidade) {
    if (quantidade <= 0) {
      throw new Error("Quantidade vendida precisa ser maior que zero.");
    }
    if (quantidade > this.estoque) {
      throw new EstoqueInsuficienteError(
        `Só tem ${this.estoque} unidade(s) de ${this.nome}. Zé da Couve tentou passar do limite.`
      );
    }

    this.estoque -= quantidade;
    return quantidade * this.preco;
  }

  descricao() {
    return `${this.nome} | R$ ${this.preco.toFixed(2)} | estoque: ${this.estoque}`;
  }
}
```

```javascript
class Fruta extends Produto {
  constructor(nome, preco, estoque, maturacao) {
    super(nome, preco, estoque);
    this.maturacao = maturacao;
  }

  descricao() {
    return `${super.descricao()} | maturação: ${this.maturacao}`;
  }
}

class Verdura extends Produto {
  constructor(nome, preco, estoque, unidade) {
    super(nome, preco, estoque);
    this.unidade = unidade;
  }

  descricao() {
    return `${super.descricao()} | vendida por: ${this.unidade}`;
  }
}

class ProdutoPromocional extends Produto {
  constructor(nome, preco, estoque, desconto) {
    super(nome, preco, estoque);
    if (desconto <= 0 || desconto >= 1) {
      throw new Error("Desconto precisa ficar entre 0 e 1.");
    }
    this.desconto = desconto;
  }

  vender(quantidade) {
    const totalNormal = super.vender(quantidade);
    return totalNormal * (1 - this.desconto);
  }

  descricao() {
    return `${super.descricao()} | desconto: ${this.desconto * 100}%`;
  }
}
```

```javascript
const manga = new Fruta("Manga do Zé da Manga", 4.5, 8, "quase no ponto");
const couve = new Verdura("Couve do Zé da Couve", 3.0, 5, "maço");
const abacaxi = new ProdutoPromocional("Abacaxi da Dona Jurema", 7.0, 3, 0.15);

const produtos = [manga, couve, abacaxi];

for (const produto of produtos) {
  console.log(produto.descricao());
}

try {
  const total = couve.vender(20);
  console.log(`Venda feita. Total: R$ ${total.toFixed(2)}`);
} catch (erro) {
  if (erro instanceof EstoqueInsuficienteError) {
    console.log("Problema de estoque:", erro.message);
  } else {
    console.log("Erro inesperado:", erro.message);
  }
} finally {
  console.log("Conferência da venda finalizada.");
}
```

## Fechamento

A graça do exemplo é perceber que os três temas se misturam:

- a análise ajuda a descobrir quais classes fazem sentido;
- o projeto organiza responsabilidades entre essas classes;
- herança e exceções deixam o comportamento mais reaproveitável e mais seguro.

No fim, POO não é só escrever `class`. É tentar fazer o código parecer com as regras do problema, sem transformar tudo numa panela só.